# INJECTION-R1 — Heads Warm-up (Colab GPU/TPU)

**Before running:**
1. Runtime → Change runtime type → **GPU (T4)** or **TPU**
2. Upload 3 files when Cell 2 prompts you:
   - `data/tokenized/ats_tokenized.npz`
   - `data/tokenized/rsg_tokenized.npz`
   - `model/unified_model/data_splits.json`
3. Run all cells in order
4. Download `r1_best_weights.h5` from Cell 9 when training completes

**Expected time:** ~1–3 min/epoch on T4 GPU → full run ~30–60 min

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 scikit-learn
print("Dependencies installed.")

In [ ]:
# ── Cell 2: Upload data files ─────────────────────────────────────────────────
# Upload: ats_tokenized.npz, rsg_tokenized.npz, data_splits.json
import os
from google.colab import files

os.makedirs('/content/data/tokenized', exist_ok=True)
os.makedirs('/content/model/unified_model', exist_ok=True)

print("Please upload the 3 required files:")
print("  1. ats_tokenized.npz  (from data/tokenized/)")
print("  2. rsg_tokenized.npz  (from data/tokenized/)")
print("  3. data_splits.json   (from model/unified_model/)")

uploaded = files.upload()

for fname, data in uploaded.items():
    if fname == 'data_splits.json':
        path = '/content/model/unified_model/data_splits.json'
    else:
        path = f'/content/data/tokenized/{fname}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f"  Saved → {path}  ({len(data)/1024:.1f} KB)")

assert os.path.exists('/content/data/tokenized/ats_tokenized.npz'), "Missing ats_tokenized.npz"
assert os.path.exists('/content/data/tokenized/rsg_tokenized.npz'), "Missing rsg_tokenized.npz"
assert os.path.exists('/content/model/unified_model/data_splits.json'), "Missing data_splits.json"
print("\nAll files uploaded OK.")

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import json
import math
import sys
import os

os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score
from transformers import TFAutoModel

# Check GPU/TPU
gpus = tf.config.list_physical_devices('GPU')
tpus = tf.config.list_physical_devices('TPU')
print(f"TF version : {tf.__version__}")
print(f"GPUs found : {len(gpus)} — {gpus}")
print(f"TPUs found : {len(tpus)} — {tpus}")

In [ ]:
# ── Cell 4: Model architecture (self-contained, no local imports) ─────────────

SEQ_LEN      = 128
EMB_DIM      = 384
NUM_DOMAINS  = 7
NUM_RSG      = 46
MINILM_NAME  = 'sentence-transformers/all-MiniLM-L6-v2'


def mean_pool_l2(last_hidden, attention_mask):
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._bert = bert_model
        self._bert.trainable = False

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=False,
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    # ATS head
    x1        = tf.keras.layers.Dense(256, activation='relu',  name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                   name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',  name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                   name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid', name='ats_score')(x1)

    # Domain head
    x2           = tf.keras.layers.Dense(256, activation='relu',  name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                   name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu',  name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                   name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax', name='domain_probs')(x2)

    # RSG head
    x3           = tf.keras.layers.Dense(512, activation='relu', name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                  name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu', name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                  name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu', name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                  name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax', name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )

print("Model architecture defined.")

In [ ]:
# ── Cell 5: Hyperparameters + load data ───────────────────────────────────────

LR         = 1e-3
BATCH_SIZE = 32
MAX_EPOCHS = 40
PATIENCE   = 8
W_ATS      = 0.35
W_DOM      = 0.35
W_RSG      = 0.30
DOM_CLS_W  = [1.4, 0.8, 0.9, 1.0, 1.5, 0.9, 1.0]

CHECKPOINT_PATH = '/content/model/unified_model/r1_best_weights.h5'

print("Loading tokenized data ...")
ats_data = np.load('/content/data/tokenized/ats_tokenized.npz')
rsg_data = np.load('/content/data/tokenized/rsg_tokenized.npz')

with open('/content/model/unified_model/data_splits.json') as f:
    splits = json.load(f)

ats_tr_idx = np.array(splits['ats_train'])
ats_vl_idx = np.array(splits['ats_val'])
rsg_tr_idx = np.array(splits['rsg_train'])
rsg_vl_idx = np.array(splits['rsg_val'])

ATS_KEYS = ('resume_input_ids', 'resume_attention_mask',
            'jd_input_ids', 'jd_attention_mask', 'ats_scores', 'domain_labels')
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_tr = {k: ats_data[k][ats_tr_idx] for k in ATS_KEYS}
ats_vl = {k: ats_data[k][ats_vl_idx] for k in ATS_KEYS}
rsg_tr = {k: rsg_data[k][rsg_tr_idx] for k in RSG_KEYS}
rsg_vl = {k: rsg_data[k][rsg_vl_idx] for k in RSG_KEYS}

n_ats_tr = len(ats_tr_idx)
n_rsg_tr = len(rsg_tr_idx)
print(f"  ATS  train={n_ats_tr:,}  val={len(ats_vl_idx):,}")
print(f"  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}")

In [ ]:
# ── Cell 6: Build model + freeze encoder ──────────────────────────────────────
print(f"Loading {MINILM_NAME} ...")
bert_model = TFAutoModel.from_pretrained(MINILM_NAME, from_pt=True)
bert_model.trainable = False

model = build_unified_minilm_model(bert_model)

for layer in model.layers:
    if 'minilm' in layer.name or 'encoder' in layer.name:
        layer.trainable = False

trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
frozen    = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
print(f"  Trainable params : {trainable:,}  (heads only)")
print(f"  Frozen params    : {frozen:,}  (MiniLM encoder)")

In [ ]:
# ── Cell 7: Loss helpers + optimizer ──────────────────────────────────────────

dom_w_tf = tf.constant(DOM_CLS_W, dtype=tf.float32)
_sce     = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)


def _ats_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - tf.squeeze(y_pred, axis=-1)))


def _dom_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    w  = tf.gather(dom_w_tf, tf.cast(y_true, tf.int32))
    return tf.reduce_mean(ce * w)


@tf.function
def train_step_ats(r_ids, r_mask, j_ids, j_mask, ats_y, dom_y):
    with tf.GradientTape() as tape:
        ats_p, dom_p, _ = model([r_ids, r_mask, j_ids, j_mask], training=True)
        l_ats = _ats_mae(ats_y, ats_p)
        l_dom = _dom_loss(dom_y, dom_p)
        loss  = W_ATS * l_ats + W_DOM * l_dom
    optimizer.apply_gradients(zip(
        tape.gradient(loss, model.trainable_variables), model.trainable_variables
    ))
    return loss, l_ats, l_dom


@tf.function
def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        l_rsg = _sce(rsg_y, rsg_p)
        loss  = W_RSG * l_rsg
    optimizer.apply_gradients(zip(
        tape.gradient(loss, model.trainable_variables), model.trainable_variables
    ))
    return loss, l_rsg


def run_validation():
    n = len(ats_vl_idx)
    ats_preds, dom_preds = [], []
    for s in range(0, n, BATCH_SIZE):
        e = min(s + BATCH_SIZE, n)
        ap, dp, _ = model(
            [ats_vl['resume_input_ids'][s:e], ats_vl['resume_attention_mask'][s:e],
             ats_vl['jd_input_ids'][s:e],     ats_vl['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_preds.append(ap.numpy())
        dom_preds.append(dp.numpy())

    ats_preds = np.concatenate(ats_preds).squeeze(-1)
    dom_preds = np.concatenate(dom_preds)
    val_mae01 = float(np.mean(np.abs(ats_preds - ats_vl['ats_scores'])))
    dom_true  = ats_vl['domain_labels']
    dom_cls   = np.argmax(dom_preds, axis=1)
    val_f1    = f1_score(dom_true, dom_cls, average='macro', zero_division=0)
    val_dom_ce = float(np.mean(
        tf.keras.losses.sparse_categorical_crossentropy(dom_true, dom_preds).numpy()
    ))

    m = len(rsg_vl_idx)
    rsg_preds = []
    for s in range(0, m, BATCH_SIZE):
        e    = min(s + BATCH_SIZE, m)
        pids = rsg_vl['profile_input_ids'][s:e]
        pmsk = rsg_vl['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_preds.append(rp.numpy())

    rsg_preds = np.concatenate(rsg_preds)
    rsg_true  = rsg_vl['rsg_labels']
    rsg_acc   = float(np.mean(np.argmax(rsg_preds, axis=1) == rsg_true))
    rsg_ce    = float(np.mean(
        tf.keras.losses.sparse_categorical_crossentropy(rsg_true, rsg_preds).numpy()
    ))

    val_loss = W_ATS * val_mae01 + W_DOM * val_dom_ce + W_RSG * rsg_ce
    return val_loss, val_mae01 * 100.0, val_f1, rsg_acc


print("Loss functions and optimizer ready.")

In [ ]:
# ── Cell 8: Training loop ─────────────────────────────────────────────────────

rng              = np.random.default_rng(42)
best_val_loss    = float('inf')
patience_counter = 0
best_epoch       = -1

ats_batches_per_epoch = math.ceil(n_ats_tr / BATCH_SIZE)
print(f"Starting training: max_epochs={MAX_EPOCHS}  batch={BATCH_SIZE}  lr={LR}")
print(f"  ATS batches/epoch: {ats_batches_per_epoch}  RSG batches/epoch: {math.ceil(n_rsg_tr/BATCH_SIZE)}")
print()

for epoch in range(MAX_EPOCHS):
    ats_perm = rng.permutation(n_ats_tr)
    rsg_perm = rng.permutation(n_rsg_tr)

    ats_batch_idxs = [ats_perm[i:i+BATCH_SIZE] for i in range(0, n_ats_tr, BATCH_SIZE)]
    rsg_batch_idxs = [rsg_perm[i:i+BATCH_SIZE] for i in range(0, n_rsg_tr, BATCH_SIZE)]
    rsg_cycle = (rsg_batch_idxs * (len(ats_batch_idxs) // len(rsg_batch_idxs) + 1))[:len(ats_batch_idxs)]

    epoch_loss = 0.0
    for aidx, ridx in zip(ats_batch_idxs, rsg_cycle):
        _, la, ld = train_step_ats(
            ats_tr['resume_input_ids'][aidx],
            ats_tr['resume_attention_mask'][aidx],
            ats_tr['jd_input_ids'][aidx],
            ats_tr['jd_attention_mask'][aidx],
            ats_tr['ats_scores'][aidx].astype(np.float32),
            ats_tr['domain_labels'][aidx],
        )
        epoch_loss += W_ATS * float(la) + W_DOM * float(ld)

        _, l_rsg = train_step_rsg(
            rsg_tr['profile_input_ids'][ridx],
            rsg_tr['profile_attention_mask'][ridx],
            rsg_tr['rsg_labels'][ridx],
        )
        epoch_loss += W_RSG * float(l_rsg)

    train_loss = epoch_loss / len(ats_batch_idxs)

    if np.isnan(train_loss):
        print(f"HARD STOP — NaN train_loss at epoch {epoch+1}")
        break

    val_loss, val_mae, val_f1, val_acc = run_validation()

    if np.isnan(val_loss):
        print(f"HARD STOP — NaN val_loss at epoch {epoch+1}")
        break

    print(f"Epoch {epoch+1:3d}/{MAX_EPOCHS}  "
          f"train={train_loss:.4f}  val={val_loss:.4f}  "
          f"ATS_MAE={val_mae:.2f}  Dom_F1={val_f1:.4f}  RSG_Acc={val_acc:.4f}")

    if epoch >= 9 and val_mae > 15.0:
        print(f"HARD STOP — ATS MAE {val_mae:.2f} > 15.0 after epoch {epoch+1}")
        break

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch + 1
        patience_counter = 0
        model.save_weights(CHECKPOINT_PATH)
        print(f"  [Saved] best weights → epoch {epoch+1}  val_loss={val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (patience={PATIENCE})")
            break

In [ ]:
# ── Cell 9: Final summary + download weights ──────────────────────────────────

model.load_weights(CHECKPOINT_PATH)
_, final_mae, final_f1, final_acc = run_validation()

print('\n' + '=' * 60)
print('  R1 HEADS WARM-UP SUMMARY')
print('=' * 60)
print(f'  Best epoch    : {best_epoch}')
print(f'  ATS val MAE   : {final_mae:.2f}  (target <10.0)  ' +
      ('PASS' if final_mae < 10.0 else 'soft miss'))
print(f'  Domain val F1 : {final_f1:.4f}  (target >0.72)  ' +
      ('PASS' if final_f1 > 0.72 else 'soft miss'))
print(f'  RSG val Acc   : {final_acc:.4f}  (target >0.42)  ' +
      ('PASS' if final_acc > 0.42 else 'soft miss'))
print('=' * 60)
print('R1 COMPLETE — proceed to R2')

# Download the weights file
from google.colab import files
files.download(CHECKPOINT_PATH)
print(f'\nDownloading {CHECKPOINT_PATH} ...')
print('Save it to: model/unified_model/r1_best_weights.h5 in your local project.')